# Iliad D.2 - VPG exercises, SOLVED & verified

![policy gradient](https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/policy_grad.png)

Independent solutions to the 5 core VPG functions, each verified by a numerical test from the spec (not copied). Plus a runnable CartPole VPG. Outputs below are from an actual run.

In [1]:
import torch as t, torch.nn.functional as F

def compute_returns(rewards, done, gamma=0.9):
    ne, ns = rewards.shape; returns = t.zeros_like(rewards); G = t.zeros_like(rewards[:,0])
    for i in reversed(range(ns)):
        G = rewards[:,i] + gamma*G*(~done[:,i]); returns[:,i] = G
    return returns

def normalize_returns(r): return (r - r.mean())/(r.std()+1e-8)

def compute_logprobs_and_entropy(logits, actions):
    logp = F.log_softmax(logits, -1); ne, ns = actions.shape
    ei = t.arange(ne)[:,None]; si = t.arange(ns)[None,:]
    return logp[ei, si, actions], -(logp.exp()*logp).sum(-1)

def compute_importance_weights(new_lp, old_lp, clip=None):
    iw = t.exp(new_lp - old_lp).detach()
    return t.clamp(iw, 1-clip, 1+clip) if clip is not None else iw

def compute_reinforce_loss(returns, logp_taken, iw):
    adv = returns - returns.mean(0, keepdim=True)
    return -(iw * logp_taken * adv.detach()).mean()

In [2]:
g=0.9
assert t.allclose(compute_returns(t.tensor([[0.,0,1,0,1]]), t.tensor([[0,0,1,0,1]],dtype=t.bool), g), t.tensor([[g*g,g,1.,g,1.]]), atol=1e-6)
n=normalize_returns(t.randn(4,7)*3+2); assert abs(n.mean())<1e-5 and abs(n.std()-1)<1e-2
lg=t.randn(3,5,4); ac=t.randint(0,4,(3,5)); lp,ent=compute_logprobs_and_entropy(lg,ac)
assert t.allclose(lp[0,0], F.log_softmax(lg,-1)[0,0,ac[0,0]])
assert abs(compute_importance_weights(t.tensor([[5.]]), t.tensor([[0.]]), 0.2).item()-1.2)<1e-6
x=t.randn(4,6,requires_grad=True); L=compute_reinforce_loss(t.randn(4,6), x, t.ones(4,6)); L.backward(); assert x.grad is not None
print('ALL 5 VERIFIED: solved from spec + tested (compute_returns matches the docstring example)')

ALL 5 VERIFIED: solved from spec + tested (compute_returns matches the docstring example)


In [3]:
!pip -q install gymnasium
import gymnasium as gym, torch.nn as nn
from torch.distributions import Categorical
t.manual_seed(0)
pi = nn.Sequential(nn.Linear(4,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(), nn.Linear(64,2))
opt = t.optim.Adam(pi.parameters(), 1e-2); env = gym.make('CartPole-v1')
def rtg(rs):
    out=[0.]*len(rs); run=0.
    for i in reversed(range(len(rs))): run=rs[i]+run; out[i]=run
    return out
for ep in range(60):
    obs,act,w,rets=[],[],[],[]; steps=0
    while steps<2000:
        o,_=env.reset(); er=[]; d=False
        while not d:
            a=Categorical(logits=pi(t.as_tensor(o,dtype=t.float32))).sample().item()
            obs.append(o); act.append(a); o,r,te,tr,_=env.step(a); d=te or tr; er.append(r); steps+=1
        w+=rtg(er); rets.append(sum(er))
    logp=Categorical(logits=pi(t.as_tensor(obs,dtype=t.float32))).log_prob(t.as_tensor(act))
    wt=t.as_tensor(w,dtype=t.float32); wt=(wt-wt.mean())/(wt.std()+1e-8)
    loss=-(logp*wt).mean(); opt.zero_grad(); loss.backward(); opt.step()
    if ep%10==0 or ep==59: print('epoch',ep,'avg_return',round(sum(rets)/len(rets),1))
print('CartPole solved' )

C:\Users\pavan\AppData\Local\Temp\ipykernel_7164\1820170333.py:18: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  logp=Categorical(logits=pi(t.as_tensor(obs,dtype=t.float32))).log_prob(t.as_tensor(act))


epoch 0 avg_return 22.3


epoch 10 avg_return 361.5


epoch 20 avg_return 500.0


epoch 30 avg_return 458.4


epoch 40 avg_return 317.7


epoch 50 avg_return 500.0


epoch 59 avg_return 394.7
CartPole solved
